# Pairwise judging (Kaggle utility benchmark)

Runs `scripts/judge_pairwise_kaggle.py` on the saved pairwise files only; no generation happens here. Create this notebook from kaggle.com/benchmarks/tasks/new so `kaggle_benchmarks` and the model proxy are available. Set `COMMIT` to the repo commit that holds the pairs files.

In [ ]:
COMMIT = "main"  # replace with the commit sha recorded in the report
REPO = "https://github.com/hafeez381/Qwen2.5-0.5B-alignment.git"
JUDGE = "google/gemini-3.6-flash"
!git clone -q $REPO /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q $COMMIT && git rev-parse HEAD

In [ ]:
import kaggle_benchmarks as kbench
print(JUDGE in kbench.llms, JUDGE)
print(kbench.llms[JUDGE].prompt('Reply with the single word: ready'))

In [ ]:
import subprocess
JOBS = [
    ("hh_dpo_vs_sft", "safety"),
    ("uf_dpo_vs_sft", "general"),
    ("lora_vs_base", "style"),
    ("lora_vs_base", "usefulness"),
]
for name, rubric in JOBS:
    for suffix in ("", "_swapped"):
        pairs = f"/kaggle/working/repo/results/eval/{name}/pairs{suffix}.jsonl"
        out = f"/kaggle/working/judged/{name}/judged_{rubric}{suffix}.jsonl"
        cmd = ["python", "/kaggle/working/repo/scripts/judge_pairwise_kaggle.py", "--pairs_path", pairs, "--output_path", out,
               "--judge_model", JUDGE, "--rubric", rubric, "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]
        print(" ".join(cmd[1:]), flush=True)
        subprocess.run(cmd, check=False)

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/judged", "zip", "/kaggle/working", "judged"))
!find /kaggle/working/judged -name '*.jsonl' | sort | xargs wc -l